# Solution D1 · How many trees do you need? Reading OOB curves, test curves and disagreement between forests

**Type**: Interpret

## Objective

Read how the OOB error, the test error and the **disagreement between two forests with different seeds** change as `n_estimators` grows, on two different problems: an "easy" one (the error settles early) and one with hard interactions (the error keeps dropping). Conclude when "a budget of 100 trees" is enough and when it is not.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X1, y1 = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
rng = np.random.default_rng(0)
X2 = rng.normal(size=(2000, 10))
y2 = ((X2[:, 0] * X2[:, 1] > 0) ^ (X2[:, 2] > 0.3)).astype(int)
y2 = np.where(rng.random(2000) < 0.05, 1 - y2, y2)
problems = {"easy": (X1, y1), "interactions": (X2, y2)}
splits = {name: train_test_split(X, y, test_size=0.4, random_state=0) for name, (X, y) in problems.items()}
ns = [5, 20, 50, 100, 250, 500]

## Commented solution

In [ ]:
oob_err = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
test_err = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    f = RandomForestClassifier(warm_start=True, oob_score=True, random_state=0, n_jobs=-1)
    for n in ns:
        f.set_params(n_estimators=n).fit(X_tr, y_tr)                     # adds trees to the ones already trained
        oob_err.loc[n, name] = 1 - f.oob_score_
        test_err.loc[n, name] = 1 - f.score(X_te, y_te)
pd.concat({"OOB": oob_err, "test": test_err}, axis=1).round(3)

In [ ]:
disagreement = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    for n in ns:
        P = [RandomForestClassifier(n, random_state=s, n_jobs=-1).fit(X_tr, y_tr).predict(X_te) for s in range(6)]
        disagreement.loc[n, name] = np.mean([(P[i] != P[j]).mean() for i in range(6) for j in range(i + 1, 6)])
disagreement.round(3)

In [ ]:
stable_n_error = {name: int(next(n for n in ns if abs(oob_err.loc[n, name] - oob_err.loc[500, name]) <= 0.01)) for name in problems}
stable_n_predictions = {name: int(next((n for n in ns if disagreement.loc[n, name] < 0.05), 10 ** 9)) for name in problems}
needs_more_trees = max(stable_n_error, key=stable_n_error.get)
predictions_still_unstable = bool(disagreement.loc[500, "interactions"] > 0.05)
print("trees to settle the OOB error within 0.01:", stable_n_error)
print("trees for a disagreement between forests under 5% (10^9 = never):", stable_n_predictions)
print("problem that needs more trees:", needs_more_trees, f"| disagreement at 500 trees in the problem with interactions: {disagreement.loc[500, 'interactions']:.3f}", "| predictions still unstable:", predictions_still_unstable)

## Check

In [ ]:
with verify():
    check(oob_err.notna().all().all() and test_err.notna().all().all() and oob_err.loc[5, "easy"] > oob_err.loc[500, "easy"] + 0.03, "complete OOB and test curves; for the easy problem the OOB error drops by over 3 points from 5 to 500 trees", "1 - oob_score_ and 1 - score(X_te, y_te)")
    check(oob_err.loc[500, "interactions"] < oob_err.loc[100, "interactions"] - 0.05 and oob_err.loc[100, "easy"] - oob_err.loc[500, "easy"] < 0.02, "with the interactions the OOB error keeps dropping by over 5 points between 100 and 500 trees; in the easy problem by less than 2", "read the two columns")
    check(disagreement.notna().all().all() and disagreement["easy"].is_monotonic_decreasing and disagreement.loc[5, "easy"] > 0.1 and disagreement.loc[500, "easy"] < 0.05, "the disagreement between forests drops: over 0.10 with 5 trees, under 0.05 with 500 (easy problem)", "mean over the pairs of the 6 forests")
    check(stable_n_error["easy"] <= 100 and stable_n_error["interactions"] >= 250, "the easy problem settles within 100 trees; the interactions need at least 250", "oob_err within 0.01 of the value at 500")
    check(needs_more_trees == "interactions" and predictions_still_unstable is True and disagreement.loc[500, "interactions"] > 0.05 and stable_n_predictions["easy"] <= 100, "the interactions need more trees and at 500 the predictions still disagree on over 5%; the easy problem has stable predictions with 100 trees or fewer", "compare the two dictionaries and the table")
done()

## What you learned

- **There is no magic number of trees.** In the easy problem the OOB error is almost flat from 100 trees on (0.128 at 100, 0.127 at 500); in the problem with interactions it keeps dropping up to 250 and beyond (from 0.45 to 0.26): the signal is hard to extract and each tree captures a fragment of it, so many trees are needed to accumulate it.
- **The predictions can stay unstable even when the mean error has stopped.** In the problem with interactions the OOB error is stable from 250 trees, but **at 500 trees two forests with different seeds still disagree on 10% of the cases** (in the easy problem the disagreement drops below 5% already at 50 trees). If you need reproducible predictions (for example the ranking of customers to contact), you need more trees than are needed for the mean accuracy.
- **Reading the OOB curve is a cheap check:** with `warm_start` you get all the points in a single incremental training. If the curve is still going down at the end, increase `n_estimators` (or revisit `max_features` and the minimum leaf size: the trees might be too weak).
- A high `n_estimators` does not cause overfitting: at worst it stops gaining. That is why you choose "high enough" and pay in time, memory and latency.

## Common pitfall

Fixing `n_estimators=100` everywhere because "it is the default". With easy problems it is plenty, with hard interaction problems it is too little. Look at the OOB curve of **your** problem.

## Interview question

<details><summary>How do you decide how many trees to use in a forest?</summary>

I look at the OOB error curve as n_estimators grows (with warm_start it is a single incremental training) and pick a number after which it is flat. If I need stability of the predictions (rankings, reproducible decisions) I also check the disagreement between two forests with different seeds, which
settles later than the mean error. The number is a budget: more trees do not overfit but cost time, memory and latency.

</details>